# Volatility Forecasting & VaR Backtesting — Narrative Walkthrough

All logic lives in `src/`; this notebook only orchestrates and narrates. It
prefers the cached walk-forward panel (`results/walkforward.pkl`, written by
`python run_analysis.py`) and falls back to recomputing if the cache is absent.

**Risk measurement, not trading.** No alpha, no P&L — just: *is the VaR model
well-calibrated out-of-sample, and where does it break?*

In [ ]:
import os, sys, pickle
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data import build_returns, TICKERS, PORTFOLIO_COL
from src import var, volatility, backtest
import run_analysis as ra

np.random.seed(42)
ASSET_ORDER = list(TICKERS) + [PORTFOLIO_COL]
pd.set_option('display.width', 200)

## 1. Data
Daily adjusted-close log returns for SPY / QQQ / TLT / GLD plus an equal-weight
portfolio, 2015–2025. Prices are cached in `data/prices.csv`.

In [ ]:
returns = build_returns()
print(f'{len(returns)} obs, {returns.index[0].date()} -> {returns.index[-1].date()}')
returns.describe().T[['mean', 'std', 'min', 'max']]

## 2. Volatility models
Rolling 250-day std, EWMA(0.94) and GARCH(1,1) conditional volatility for SPY
against a 21-day realised proxy (annualised %). *This full-sample fit is for
intuition only — the VaR backtest below is strictly walk-forward.*

In [ ]:
ann = np.sqrt(252) * 100
r = returns['SPY']
fig, ax = plt.subplots(figsize=(11, 4))
(r.rolling(21).std() * ann).plot(ax=ax, color='0.6', lw=0.8, label='Realised (21d)')
(volatility.rolling_std(r, 250) * ann).plot(ax=ax, lw=1, label='Rolling 250d')
(volatility.ewma_vol(r) * ann).plot(ax=ax, lw=1, label='EWMA 0.94')
(volatility.garch_forecast(r, 'normal') * ann).plot(ax=ax, lw=1, label='GARCH-N')
ax.set_title('SPY annualised volatility (%)'); ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 3. Walk-forward VaR + backtest
Load the cached panel (or recompute). Each forecast at day *t* uses only data
through *t-1* (trailing 500-day window).

In [ ]:
pkl = os.path.join('..', 'results', 'walkforward.pkl')
if os.path.exists(pkl):
    with open(pkl, 'rb') as fh:
        results = pickle.load(fh)
    print('Loaded cached walk-forward panel.')
else:
    print('No cache — recomputing (a few minutes)...')
    results = ra.run_all_backtests(returns, list(var.METHODS), var.GARCH_REFIT_EVERY)

summary = ra.build_summary(results)
summary.head()

### 99% VaR — the regulatory headline
Expected breach rate 1%. Watch the parametric-normal models over-breach and
GARCH-t pull closest to target.

In [ ]:
order = ['normal_rolling', 'ewma', 'historical', 'garch_normal', 'garch_t']
sub = summary[summary.confidence == 0.99]
brk = sub.pivot_table(index='model', columns='asset', values='breach_rate').reindex(order)
(brk[ASSET_ORDER] * 100).round(2)

In [ ]:
# Kupiec pass (fail-to-reject at 5%) and Basel zone at 99%
kp = sub.pivot_table(index='model', columns='asset', values='kupiec_p').reindex(order)[ASSET_ORDER]
print('Kupiec p-values (99%):'); display(kp.round(3))
tl = sub.pivot_table(index='model', columns='asset', values='traffic_light',
                     aggfunc='first').reindex(order)[ASSET_ORDER]
print('Basel traffic light (99%):'); display(tl)

## 4. Breach-rate comparison & VaR band
Mean breach rate across assets vs. target, and the SPY return series with the
99% GARCH-t VaR band and breach days in red.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, conf in zip(axes, (0.95, 0.99)):
    s = summary[summary.confidence == conf]
    m = s.groupby('model')['breach_rate'].mean().reindex(order) * 100
    ax.bar([ra.MODEL_LABELS[k] for k in m.index], m.values, color='tab:blue', alpha=.8)
    ax.axhline((1 - conf) * 100, color='red', ls='--', label=f'Expected {int((1-conf)*100)}%')
    ax.set_title(f'{int(conf*100)}% VaR'); ax.tick_params(axis='x', rotation=30)
    ax.legend(); ax.grid(alpha=.3, axis='y')
plt.tight_layout(); plt.show()

In [ ]:
wf = results['SPY']['garch_t']
br = backtest.exceedances(wf['actual'], wf['var_0.99'])
fig, ax = plt.subplots(figsize=(11, 4))
(wf['actual'] * 100).plot(ax=ax, color='0.5', lw=0.5, label='SPY return %')
(-wf['var_0.99'] * 100).plot(ax=ax, color='tab:blue', lw=1, label='99% VaR')
pts = (wf['actual'] * 100)[br.values]
ax.scatter(pts.index, pts, color='red', s=12, zorder=5, label=f'Breach (n={int(br.sum())})')
ax.set_title('SPY — GARCH(1,1)-t, 99% VaR'); ax.legend(loc='lower left'); ax.grid(alpha=.3)
plt.show()

## 6. Conclusion
**At 99% VaR, GARCH(1,1)-t wins.** It roughly halves the exceedance rate of the
normal-tailed models, is the only model simultaneously well-calibrated
(fat-enough tail → Kupiec) and free of breach clustering (volatility adaptivity
→ Christoffersen), **and is the most stable across regimes** (lowest worst-regime
breach rate, tightest cross-regime spread). **Honest caveat:** it still
over-breaches in the tails — ~1.5% on the equity indices and up to ~2.9% in the
COVID shock — so the win is a *graceful* failure, not a solved problem. At 95%
every model is adequate and the extra machinery doesn't earn its keep. See
`README.md` for the full write-up and limitations.

In [ ]:
from src import regimes
cal = regimes.regime_table(results, returns, 0.99, 'calendar')
vs  = regimes.regime_table(results, returns, 0.99, 'volstate')
print('Breach rate % by calendar regime (mean across assets, 99%):')
display(regimes.breach_rate_matrix(cal, regimes.CALENDAR_ORDER).reindex(order).round(2))
print('\nBreach rate % by volatility state (99%):')
display(regimes.breach_rate_matrix(vs, regimes.VOL_STATE_ORDER).reindex(order).round(2))
print('\nStability leaderboard (99%, lower = more stable across regimes):')
regimes.stability_leaderboard(cal, 0.99, regimes.CALENDAR_ORDER).round(2)

## 5. Conclusion
**At 99% VaR, GARCH(1,1)-t wins.** It roughly halves the exceedance rate of the
normal-tailed models and is the only model that is simultaneously well-calibrated
(fat-enough tail → Kupiec) and free of breach clustering (volatility adaptivity
→ Christoffersen), giving it the best Basel traffic-light record. **Honest
caveat:** it still slightly over-breaches on the fat-tailed equity indices
(SPY/QQQ). At 95% every model is adequate and the extra machinery doesn't earn
its keep. See `README.md` for the full write-up and limitations.